# Model evaluation

Compare several classifiers on the exact train/test split exported by `data_cleaning.ipynb`. Run the final save cell in that notebook first to create `data/train.parquet` and `data/test.parquet`.

## Imports and configuration

In [12]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA_DIR = Path("../data")
FEATURE_COLUMNS = [
    "idstation",
    "lat",
    "lon",
    "slots_total",
    "hour",
    "day_of_week",
    "is_weekend",
]
TARGET_COLUMN = "bike_bin"

## Load the saved train and test data

These files come directly from the `train_test_split` call in the cleaning notebook; this notebook does not recreate or change the split.

In [14]:
train_data = pd.read_parquet(DATA_DIR / "train.parquet")
test_data = pd.read_parquet(DATA_DIR / "test.parquet")
required_columns = [*FEATURE_COLUMNS, TARGET_COLUMN]

for split_name, split_data in (("train", train_data), ("test", test_data)):
    missing_columns = sorted(set(required_columns) - set(split_data.columns))
    if missing_columns:
        raise ValueError(f"{split_name} data is missing columns: {missing_columns}")
    if split_data.empty:
        raise ValueError(f"{split_name} data is empty.")
    if split_data[required_columns].isna().any().any():
        raise ValueError(f"{split_name} data contains missing model inputs or targets.")

X_train = train_data[FEATURE_COLUMNS]
y_train = train_data[TARGET_COLUMN]
X_test = test_data[FEATURE_COLUMNS]
y_test = test_data[TARGET_COLUMN]

print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
display(
    pd.DataFrame(
        {
            "train": y_train.value_counts(normalize=True).sort_index(),
            "test": y_test.value_counts(normalize=True).sort_index(),
        }
    ).fillna(0).round(3)
)

Training rows: 10,345
Test rows: 2,587


,train,test
bike_bin,,
0,0.041,0.041
1,0.115,0.115
2,0.217,0.217
3,0.627,0.627


## Train and compare models

Logistic regression uses standardized features. `train_accuracy` and `test_accuracy` show accuracy on the fitted and held-out data. `test_error_rate` is `1 - test_accuracy`; `generalization_gap` is `train_accuracy - test_accuracy` (a larger positive gap can indicate overfitting). Macro F1 gives each class equal weight and is useful for this imbalanced target.

In [15]:
models = {
    "Logistic regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42,
        ),
    ),
    "Decision tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42,
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ),
}

majority_class = y_train.mode().iloc[0]
predictions_by_model = {
    "Majority-class baseline": pd.Series(majority_class, index=y_test.index)
}
train_predictions_by_model = {
    "Majority-class baseline": pd.Series(majority_class, index=y_train.index)
}

for model_name, estimator in models.items():
    estimator.fit(X_train, y_train)
    train_predictions_by_model[model_name] = estimator.predict(X_train)
    predictions_by_model[model_name] = estimator.predict(X_test)

comparison = pd.DataFrame(
    {
        "train_accuracy": {
            name: accuracy_score(y_train, predictions)
            for name, predictions in train_predictions_by_model.items()
        },
        "test_accuracy": {
            name: accuracy_score(y_test, predictions)
            for name, predictions in predictions_by_model.items()
        },
        "test_error_rate": {
            name: 1 - accuracy_score(y_test, predictions)
            for name, predictions in predictions_by_model.items()
        },
        "generalization_gap": {
            name: accuracy_score(y_train, train_predictions_by_model[name])
            - accuracy_score(y_test, predictions)
            for name, predictions in predictions_by_model.items()
        },
        "macro_f1": {
            name: f1_score(y_test, predictions, average="macro", zero_division=0)
            for name, predictions in predictions_by_model.items()
        },
    }
).sort_values("test_accuracy", ascending=False)
display(comparison.round(3))

,train_accuracy,test_accuracy,test_error_rate,generalization_gap,macro_f1
Random forest,0.872,0.766,0.234,0.106,0.647
Decision tree,0.862,0.765,0.235,0.097,0.648
Majority-class baseline,0.627,0.627,0.373,-0.000,0.193
Logistic regression,0.343,0.336,0.664,0.007,0.254


## Inspect the random forest results

In [16]:
forest_predictions = predictions_by_model["Random forest"]
class_labels = sorted(set(y_train.unique()) | set(y_test.unique()))

print("Random forest classification report:")
print(
    classification_report(
        y_test,
        forest_predictions,
        labels=class_labels,
        target_names=[str(label) for label in class_labels],
        zero_division=0,
    )
)

print("Random forest confusion matrix (rows: actual, columns: predicted):")
display(
    pd.DataFrame(
        confusion_matrix(y_test, forest_predictions, labels=class_labels),
        index=pd.Index(class_labels, name="actual"),
        columns=pd.Index(class_labels, name="predicted"),
    )
)

feature_importance = pd.Series(
    models["Random forest"].feature_importances_,
    index=FEATURE_COLUMNS,
    name="importance",
).sort_values(ascending=False)
display(feature_importance.to_frame())

Random forest classification report:
              precision    recall  f1-score   support

           0       0.44      0.64      0.52       107
           1       0.51      0.62      0.56       297
           2       0.62      0.64      0.63       561
           3       0.92      0.84      0.88      1622

    accuracy                           0.77      2587
   macro avg       0.62      0.69      0.65      2587
weighted avg       0.79      0.77      0.77      2587

Random forest confusion matrix (rows: actual, columns: predicted):


predicted,0,1,2,3
actual,,,,
0,68,32,5,2
1,45,184,55,13
2,16,82,361,102
3,26,64,163,1369


,importance
hour,0.367878
day_of_week,0.245094
slots_total,0.109898
idstation,0.093764
lon,0.079456
lat,0.077170
is_weekend,0.026740
